# FER-2013: model comparison with an 80/10/10 split

Download the complete [FER-2013 dataset](https://www.kaggle.com/datasets/msambare/fer2013), combine Kaggle's original `train` and `test` directories, create deterministic class-stratified splits, and compare **VGG16** and **ResNet50** under the same training and evaluation protocol. The manifests are saved without duplicating image files.

## 1. Setup

Install dependencies with `pip install -r requirements.txt` and select the project `.venv` as the notebook kernel.

In [ ]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
import tensorflow as tf

DATASET_HANDLE = "msambare/fer2013"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
VALIDATION_COUNTS = {
    "angry": 467, "disgust": 56, "fear": 496, "happy": 895,
    "sad": 653, "surprise": 415, "neutral": 607,
}
LABEL_EN = {
    "angry": "Angry", "disgust": "Disgust", "fear": "Fear",
    "happy": "Happy", "sad": "Sad",
    "surprise": "Surprise", "neutral": "Neutral",
}
SEED = 42
print(f"TensorFlow version: {tf.__version__}")

## 2. Download the complete dataset

KaggleHub downloads both original directories and reuses its cache on later runs.

In [3]:
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
source_dirs = {name: dataset_root / name for name in ("train", "test")}
missing = [name for name, path in source_dirs.items() if not path.is_dir()]
if missing:
    available = sorted(path.name for path in dataset_root.iterdir())
    raise FileNotFoundError(f"Missing {missing} under {dataset_root}. Found: {available}")

print(f"Dataset cache: {dataset_root}")
for name, path in source_dirs.items():
    print(f"Original {name:>5} directory: {path}")

100%|██████████| 60.3M/60.3M [00:05<00:00, 12.0MB/s]

Extracting files...


Dataset cache: C:\Users\PC\.cache\kagglehub\datasets\msambare\fer2013\versions\1
Original train directory: C:\Users\PC\.cache\kagglehub\datasets\msambare\fer2013\versions\1\train
Original  test directory: C:\Users\PC\.cache\kagglehub\datasets\msambare\fer2013\versions\1\test


## 3. Inventory every image

Count all valid images and display the planned training, validation, and test totals per emotion. The validation counts follow the configured class counts; section 4 assigns individual images to these splits.


In [ ]:
records, errors = [], []
for source_split, source_dir in source_dirs.items():
    for class_dir in sorted(path for path in source_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.rglob("*")):
            if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            try:
                with Image.open(image_path) as image:
                    width, height = image.size
                    mode = image.mode
                    image.verify()
                records.append({
                    "path": str(image_path),
                    "relative_path": str(image_path.relative_to(dataset_root)),
                    "label": class_dir.name,
                    "source_split": source_split,
                    "width": width, "height": height, "mode": mode,
                    "size_bytes": image_path.stat().st_size,
                })
            except Exception as exc:
                errors.append({"path": str(image_path), "error": str(exc)})

image_info = pd.DataFrame.from_records(records)
if image_info.empty:
    raise RuntimeError(f"No supported images found in {dataset_root}")
print(f"Valid images: {len(image_info):,}; unreadable files: {len(errors)}")
inventory_counts = pd.crosstab(image_info["label"], image_info["source_split"])
inventory_summary = pd.DataFrame({
    "train": inventory_counts["train"],
    "validation": pd.Series(VALIDATION_COUNTS),
})
inventory_summary["test"] = inventory_counts["test"] - inventory_summary["validation"]
assert inventory_summary.notna().all().all()
assert (inventory_summary >= 0).all().all()
inventory_summary = inventory_summary.reindex([
    "angry", "disgust", "fear", "happy", "sad", "surprise", "neutral",
])
inventory_summary = inventory_summary.rename(index=LABEL_EN, columns={
    "train": "Training",
    "validation": "Validation",
    "test": "Test",
})
inventory_summary["Total"] = inventory_summary.sum(axis=1)
inventory_summary.loc["Total"] = inventory_summary.sum()
inventory_summary = inventory_summary.astype(int)
inventory_summary.index.name = None
inventory_summary.columns.name = None
display(inventory_summary.map(lambda value: f"{value:,}"))

## 4. Create the 80/10/10 splits

Keep all 28,709 images from Kaggle's original `train` directory for training. Shuffle the original `test` images within each class and divide them into validation and test sets using the requested class counts. This gives exactly 28,709 / 3,589 / 3,589 images.

In [ ]:
rng = np.random.default_rng(SEED)
image_info["split"] = pd.Series(index=image_info.index, dtype="string")
image_info.loc[image_info["source_split"] == "train", "split"] = "train"

original_test = image_info.loc[image_info["source_split"] == "test"]
for label, group in original_test.groupby("label", sort=True):
    indices = group.index.to_numpy().copy()
    rng.shuffle(indices)
    n_validation = VALIDATION_COUNTS[label]
    image_info.loc[indices[:n_validation], "split"] = "validation"
    image_info.loc[indices[n_validation:], "split"] = "test"

assert image_info["split"].notna().all()
assert image_info["path"].is_unique
assert image_info["split"].value_counts().to_dict() == {"train": 28709, "validation": 3589, "test": 3589}

split_counts = pd.crosstab(image_info["label"], image_info["split"])[["train", "validation", "test"]]
split_counts = split_counts.rename(index=LABEL_EN, columns={"train": "Training", "validation": "Validation", "test": "Test"})
split_counts = split_counts.reindex(["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"])
split_counts["Total"] = split_counts.sum(axis=1)
split_counts.loc["Total"] = split_counts.sum()
display(split_counts.map(lambda value: f"{value:,}"))

## 5. Save manifests

Save the combined inventory and one CSV file for each new split under `data/`.

In [6]:
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)
all_path = output_dir / "fer2013_all_manifest.csv"
image_info.to_csv(all_path, index=False)
print(f"Saved {len(image_info):,} rows -> {all_path.resolve()}")
for split_name in ("train", "validation", "test"):
    frame = image_info.loc[image_info["split"] == split_name]
    split_path = output_dir / f"fer2013_{split_name}_manifest.csv"
    frame.to_csv(split_path, index=False)
    print(f"Saved {len(frame):,} rows -> {split_path.resolve()}")

Saved 35,887 rows -> C:\code\iuh-master\deep-learning\data\fer2013_all_manifest.csv
Saved 28,709 rows -> C:\code\iuh-master\deep-learning\data\fer2013_train_manifest.csv
Saved 3,589 rows -> C:\code\iuh-master\deep-learning\data\fer2013_validation_manifest.csv
Saved 3,589 rows -> C:\code\iuh-master\deep-learning\data\fer2013_test_manifest.csv


## 6. Visualize total data

In [ ]:
plot_counts = image_info["label"].value_counts().sort_index()
ax = plot_counts.plot.bar(figsize=(11, 5), color="#4C78A8")
total_images = int(plot_counts.sum())
ax.set(title=f"FER-2013 total images per emotion (Total: {total_images:,})",
       xlabel="Emotion", ylabel="Total images")
ax.bar_label(ax.containers[0], labels=[f"{count:,}" for count in plot_counts], padding=3)
ax.margins(y=0.12)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 7. Build TensorFlow datasets

Decode images as normalized 96×96 grayscale tensors. Only the training dataset is shuffled. The configured batch size is 64.

In [ ]:
BATCH_SIZE = 64
IMAGE_SIZE = (96, 96)
LEARNING_RATE = 1e-4  # Assumed because the supplied value was not visible
TOTAL_EPOCHS = 30
EARLY_STOP_PATIENCE = 7
class_names = sorted(image_info["label"].unique())
label_to_index = {label: index for index, label in enumerate(class_names)}

def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=1, expand_animations=False)
    image.set_shape([None, None, 1])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.one_hot(label, depth=len(class_names))

def make_dataset(split_name, shuffle=False):
    frame = image_info.loc[image_info["split"] == split_name]
    paths = frame["path"].to_numpy()
    labels = frame["label"].map(label_to_index).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset("train", shuffle=True)
validation_dataset = make_dataset("validation")
test_dataset = make_dataset("test")
images, labels = next(iter(train_dataset))
print(f"Classes: {class_names}")
print(f"Image batch: {images.shape}; label batch: {labels.shape}")
print(f"Pixel range: [{tf.reduce_min(images).numpy():.3f}, {tf.reduce_max(images).numpy():.3f}]")
display(pd.DataFrame({
    "Hyperparameter": ["Input size", "Batch size", "Learning rate", "Optimizer", "Epochs", "Loss function", "Early stop"],
    "Value": ["96 × 96", BATCH_SIZE, LEARNING_RATE, "Adam", TOTAL_EPOCHS, "Categorical Cross-Entropy", f"{EARLY_STOP_PATIENCE} epochs"],
}))

## 8. Data augmentation

Apply horizontal flips, rotations up to 18 degrees, 10% zoom, 5% translations, and 10% contrast adjustments. Keep pixel values in [0, 1]. Both models use the same augmentation settings during training; validation and test images remain unchanged. Augmented images are generated on the fly without saving additional image files.


In [ ]:
def make_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED + 1),
        tf.keras.layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED + 2),
        tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED + 3),
        tf.keras.layers.RandomContrast(0.10, seed=SEED + 4),
        tf.keras.layers.ReLU(max_value=1.0),
    ], name="augmentation")

preview_augmentation = make_augmentation()
preview_images, _ = next(iter(train_dataset))
preview_image = preview_images[:1]
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(preview_image[0, ..., 0].numpy(), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original")
for index, axis in enumerate(axes[1:], start=1):
    augmented_image = preview_augmentation(preview_image, training=True)
    axis.imshow(augmented_image[0, ..., 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"Augmented {index}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()
del preview_augmentation


## 9. Build VGG16 and ResNet50

VGG16 and ResNet50 use frozen ImageNet backbones: each grayscale image is copied to three channels and converted with the matching preprocessing function. Both models share the same augmentation, optimizer, learning rate, classifier size, and output classes so their results are directly comparable.
The complexity table reports total, trainable, and non-trainable parameters plus estimated inference FLOPs and GFLOPs for one 96 x 96 grayscale image (batch size 1). Multiply and add count as two operations. Training-only augmentation and dropout are disabled. FLOPs cover operations supported by the TensorFlow profiler and exclude backpropagation.


In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def build_transfer_model(application, preprocess_input, model_name):
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 1), name="grayscale_image")
    x = make_augmentation()(inputs)
    x = tf.keras.layers.Concatenate(axis=-1, name="grayscale_to_rgb")([x, x, x])
    x = tf.keras.layers.Rescaling(255.0, name="restore_pixel_range")(x)
    x = preprocess_input(x)
    backbone = application(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMAGE_SIZE, 3),
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax", name="emotion")(x)
    return compile_model(tf.keras.Model(inputs, outputs, name=model_name))

def build_vgg16():
    return build_transfer_model(
        tf.keras.applications.VGG16,
        tf.keras.applications.vgg16.preprocess_input,
        "fer2013_vgg16",
    )

def build_resnet50():
    return build_transfer_model(
        tf.keras.applications.ResNet50,
        tf.keras.applications.resnet50.preprocess_input,
        "fer2013_resnet50",
    )

MODEL_BUILDERS = {
    "VGG16": build_vgg16,
    "ResNet50": build_resnet50,
}

def estimate_flops(model):
    # Freeze inference variables and profile one forward pass for one image.
    from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

    inference = tf.function(lambda inputs: model(inputs, training=False))
    concrete = inference.get_concrete_function(
        tf.TensorSpec((1, *IMAGE_SIZE, 1), tf.float32)
    )
    frozen = convert_variables_to_constants_v2(concrete)
    options = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    options["output"] = "none"
    profile = tf.compat.v1.profiler.profile(
        graph=frozen.graph, cmd="op", options=options,
    )
    if profile is None or profile.total_float_ops <= 0:
        raise RuntimeError("Could not estimate inference FLOPs")
    return int(profile.total_float_ops)

def layer_shape_table(model):
    def shape_text(tensors):
        shapes = [str(tuple(tensor.shape)) for tensor in tf.nest.flatten(tensors)]
        return shapes[0] if len(shapes) == 1 else "[" + ", ".join(shapes) + "]"

    return pd.DataFrame([
        {
            "Layer": layer.name,
            "Type": type(layer).__name__,
            "Input shape": shape_text(layer.input),
            "Output shape": shape_text(layer.output),
            "Parameters": layer.count_params(),
        }
        for layer in model.layers
    ])

complexity_rows = []
for name, builder in MODEL_BUILDERS.items():
    tf.keras.backend.clear_session()
    candidate = builder()
    print(f"\n{name}: layer input/output shapes (None = variable batch size)")
    architecture_table = layer_shape_table(candidate)
    architecture_table.to_csv(MODEL_DIR / f"fer2013_{name.lower()}_architecture.csv", index=False)
    display(architecture_table.style.format({"Parameters": "{:,.0f}"}))
    trainable_count = int(sum(np.prod(weight.shape) for weight in candidate.trainable_weights))
    flops = estimate_flops(candidate)
    complexity_rows.append({
        "Model": name,
        "Parameters": candidate.count_params(),
        "Trainable parameters": trainable_count,
        "Non-trainable parameters": candidate.count_params() - trainable_count,
        "FLOPs": flops,
        "GFLOPs": flops / 1e9,
    })
    del candidate

tf.keras.backend.clear_session()
model_complexity = pd.DataFrame(complexity_rows)
model_complexity.to_csv(MODEL_DIR / "fer2013_model_complexity.csv", index=False)
print(f"Inference complexity per image: {IMAGE_SIZE[0]} x {IMAGE_SIZE[1]} grayscale, batch size 1")
display(model_complexity.style.format({
    "Parameters": "{:,.0f}",
    "Trainable parameters": "{:,.0f}",
    "Non-trainable parameters": "{:,.0f}",
    "FLOPs": "{:,.0f}",
    "GFLOPs": "{:.3f}",
}))


## 10. Train all models

Each model sees the same shuffled training dataset and validation dataset. Class weights reduce the effect of FER-2013's imbalance. A separate best checkpoint and CSV log are written for every model. Early stopping can end a run before 30 epochs.

In [ ]:
import time

train_counts = image_info.loc[image_info["split"] == "train", "label"].value_counts()
n_train = int(train_counts.sum())
class_weight = {
    label_to_index[label]: n_train / (len(class_names) * count)
    for label, count in train_counts.items()
}
print("Class weights:", {class_names[index]: round(weight, 3) for index, weight in class_weight.items()})

histories = {}
model_metadata = {}

for model_name, builder in MODEL_BUILDERS.items():
    print(f"\n{'=' * 24} {model_name} {'=' * 24}")
    tf.keras.backend.clear_session()
    model = builder()
    slug = model_name.lower()
    best_model_path = MODEL_DIR / f"fer2013_{slug}_best.keras"
    training_log_path = MODEL_DIR / f"fer2013_{slug}_training.csv"
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            best_model_path, monitor="val_accuracy", mode="max",
            save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2,
            min_lr=1e-7, verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(training_log_path),
    ]
    started_at = time.perf_counter()
    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=TOTAL_EPOCHS,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=2,
    )
    histories[model_name] = history.history
    model_metadata[model_name] = {
        "path": best_model_path,
        "parameters": model.count_params(),
        "trainable_parameters": int(sum(np.prod(weight.shape) for weight in model.trainable_weights)),
        "epochs": len(history.history["loss"]),
        "training_minutes": (time.perf_counter() - started_at) / 60,
    }
    del model

tf.keras.backend.clear_session()


## 11. Compare training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for model_name, metrics in histories.items():
    epochs = range(1, len(metrics["loss"]) + 1)
    axes[0].plot(epochs, metrics["val_loss"], label=model_name)
    axes[1].plot(epochs, metrics["val_accuracy"], label=model_name)

axes[0].set(title="Validation loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()


## 12. Evaluate and compare on the test set

The saved checkpoint with the highest validation accuracy is loaded for each architecture. The test set is used only for this final evaluation, not for model selection.

In [ ]:
true_labels = np.concatenate([
    np.argmax(batch_labels.numpy(), axis=1)
    for _, batch_labels in test_dataset
])

comparison_rows = []
confusion_matrices = {}

for model_name, metadata in model_metadata.items():
    print(f"\nEvaluating {model_name}")
    tf.keras.backend.clear_session()
    best_model = tf.keras.models.load_model(metadata["path"])
    test_loss, test_accuracy = best_model.evaluate(test_dataset, verbose=1)
    probabilities = best_model.predict(test_dataset, verbose=1)
    predicted_labels = np.argmax(probabilities, axis=1)
    confusion_matrices[model_name] = tf.math.confusion_matrix(
        true_labels, predicted_labels, num_classes=len(class_names)
    ).numpy()
    comparison_rows.append({
        "Model": model_name,
        "Parameters": metadata["parameters"],
        "Trainable parameters": metadata["trainable_parameters"],
        "Epochs trained": metadata["epochs"],
        "Training time (min)": metadata["training_minutes"],
        "Best validation accuracy": max(histories[model_name]["val_accuracy"]),
        "Test loss": test_loss,
        "Test accuracy": test_accuracy,
    })
    del best_model

comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("Best validation accuracy", ascending=False)
    .reset_index(drop=True)
)
comparison.to_csv(MODEL_DIR / "fer2013_model_comparison.csv", index=False)
display(comparison.style.format({
    "Parameters": "{:,.0f}",
    "Trainable parameters": "{:,.0f}",
    "Training time (min)": "{:.1f}",
    "Best validation accuracy": "{:.2%}",
    "Test loss": "{:.4f}",
    "Test accuracy": "{:.2%}",
}))
best_name = comparison.loc[0, "Model"]
print(f"Selected by validation accuracy: {best_name}")


In [ ]:
classification_rows = []
for model_name in ("VGG16", "ResNet50"):
    matrix = confusion_matrices[model_name]
    true_positive = np.diag(matrix).astype(float)
    precision = np.divide(true_positive, matrix.sum(axis=0),
                          out=np.zeros_like(true_positive), where=matrix.sum(axis=0) != 0)
    recall = np.divide(true_positive, matrix.sum(axis=1),
                       out=np.zeros_like(true_positive), where=matrix.sum(axis=1) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros_like(true_positive), where=(precision + recall) != 0)
    classification_rows.append({
        "Model": model_name,
        "Accuracy": true_positive.sum() / matrix.sum(),
        "Precision": precision.mean(),
        "Recall": recall.mean(),
        "F1-Score": f1.mean(),
    })
classification_metrics = pd.DataFrame(classification_rows).set_index("Model")
classification_metrics.to_csv(MODEL_DIR / "fer2013_classification_metrics.csv")
print("Test-set metrics; Precision, Recall, and F1-Score use macro averaging across 7 classes.")
display(classification_metrics.style.format("{:.2%}"))


In [ ]:
from matplotlib.ticker import PercentFormatter

# Use current evaluation results, or the saved table when running this cell alone.
if "classification_metrics" in globals():
    metrics_to_plot = classification_metrics.copy()
else:
    metrics_to_plot = pd.read_csv(
        Path("models") / "fer2013_classification_metrics.csv", index_col="Model",
    )
metrics_to_plot = metrics_to_plot.loc[
    ["VGG16", "ResNet50"], ["Accuracy", "Precision", "Recall", "F1-Score"],
]
ax = metrics_to_plot.T.plot.bar(
    figsize=(10, 5), color=["#4C78A8", "#E45756"], width=0.75,
)
for container in ax.containers:
    ax.bar_label(container, labels=[f"{bar.get_height():.2%}" for bar in container], padding=3)
ax.set(
    title="VGG16 vs ResNet50: test-set classification metrics",
    xlabel="Metric", ylabel="Score", ylim=(0, 1),
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.tick_params(axis="x", labelrotation=0)
ax.legend(title="Model", loc="upper right")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
ax.text(
    0.5, -0.18, "Precision, Recall, and F1-Score: macro average across 7 classes",
    transform=ax.transAxes, ha="center", fontsize=9,
)
plt.tight_layout()
figure_path = Path("models") / "fer2013_classification_metrics.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
ax.figure.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()


## 13. Compare confusion matrices

In [ ]:
fig, axes = plt.subplots(
    1, len(confusion_matrices), figsize=(12, 5), layout="compressed",
)
fig.get_layout_engine().set(w_pad=0.02, wspace=0.02)

for axis, (model_name, confusion) in zip(axes, confusion_matrices.items()):
    image = axis.imshow(confusion, cmap="Blues")
    axis.set(
        xticks=range(len(class_names)),
        yticks=range(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted label",
        ylabel="True label",
        title=model_name,
    )
    plt.setp(axis.get_xticklabels(), rotation=45, ha="right")
    threshold = confusion.max() / 2
    for row in range(len(class_names)):
        for column in range(len(class_names)):
            axis.text(
                column, row, confusion[row, column],
                ha="center", va="center", fontsize=7,
                color="white" if confusion[row, column] > threshold else "black",
            )
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.02)

plt.show()


## 14. Example: test an image

Use the model selected by validation accuracy to predict the emotion of one test image. The image, true label, predicted label, and class probabilities are displayed below.


In [ ]:
example_row = image_info.loc[image_info["split"] == "test"].sample(
    n=1, random_state=SEED,
).iloc[0]
example_path = Path(example_row["path"])
example_image, _ = load_image(
    str(example_path), label_to_index[example_row["label"]],
)
example_model = tf.keras.models.load_model(model_metadata[best_name]["path"])
example_probabilities = example_model.predict(example_image[None, ...], verbose=0)[0]
predicted_index = int(np.argmax(example_probabilities))
predicted_emotion = class_names[predicted_index]
confidence = float(example_probabilities[predicted_index])

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_image[..., 0].numpy(), cmap="gray", vmin=0, vmax=1)
ax.set_title(
    f"{best_name}\nTrue: {example_row['label']}\n"
    f"Predicted: {predicted_emotion} ({confidence:.2%})"
)
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Image: {example_path}")
display(
    pd.DataFrame({"Emotion": class_names, "Probability": example_probabilities})
    .sort_values("Probability", ascending=False)
    .reset_index(drop=True)
    .style.format({"Probability": "{:.2%}"})
)
del example_model
